# PDF RAG Pipeline — LlamaParse + OpenAI
### Keys needed: `LLAMA_CLOUD_API_KEY` · `OPENAI_API_KEY`

**What this notebook does (step by step):**
1. **Parse PDFs** with LlamaParse — handles tables, columns, headers cleanly
2. **Build a VectorStoreIndex** from the parsed text
3. **Compare embedding models**: `text-embedding-3-small` (OpenAI API) vs `BAAI/bge-large-en-v1.5` (local, free)
4. **Compare LLMs** at synthesis time: `gpt-4o-mini` vs `gpt-3.5-turbo`
5. **Evaluate** every combination: faithfulness, relevancy, latency, error rate — judged by `gpt-4o-mini`


## 0 · Install dependencies

In [1]:
!pip install -q \
    llama-index \
    llama-parse \
    llama-index-embeddings-openai \
    llama-index-embeddings-huggingface \
    llama-index-llms-openai \
    sentence-transformers \
    pandas nest_asyncio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.0/362.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 8.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


## 1 · API keys

| Key | Where to get it | Used for |
|-----|----------------|---------|
| `LLAMA_CLOUD_API_KEY` | [cloud.llamaindex.ai](https://cloud.llamaindex.ai) (free tier) | LlamaParse PDF parsing |
| `OPENAI_API_KEY` | [platform.openai.com/api-keys](https://platform.openai.com/api-keys) | Embeddings, LLM synthesis, evaluation judge |

BGE embeddings run **100% locally** via `sentence-transformers` — no key, no cost.


In [2]:
from google.colab import userdata
import os

os.environ["LLAMA_CLOUD_API_KEY"] = userdata.get("llama")
os.environ["OPENAI_API_KEY"]      = userdata.get("openAI")

print("Keys loaded:")
for k in ["LLAMA_CLOUD_API_KEY", "OPENAI_API_KEY"]:
    val = os.environ.get(k, "")
    print(f"  {k}: {'✓' if val else '✗ MISSING'}  ({val[:8]}...)" if val else f"  {k}: ✗ MISSING")

Keys loaded:
  LLAMA_CLOUD_API_KEY: ✓  (llx-zE92...)
  OPENAI_API_KEY: ✓  (sk-proj-...)


## 2 · Configuration

In [3]:
from pathlib import Path

# Directories
DATA_DIR    = Path("data")
PARSED_DIR  = Path("parsed")
INDEX_DIR   = Path("indexes")
RESULTS_DIR = Path("results")
for d in (DATA_DIR, PARSED_DIR, INDEX_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Step 3: embedding models to compare ──────────────────────────────────────
EMBEDDING_MODELS = {
    "openai_small": {
        "type": "openai",
        "model": "text-embedding-3-small",   # 1536-dim, fast, low cost
    },
    "bge_large": {
        "type": "local",
        "model": "BAAI/bge-large-en-v1.5",   # 1024-dim, local, free
    },
}

# ── Step 4: LLMs to compare at synthesis time ─────────────────────────────────
LLM_MODELS = {
    "gpt4o_mini":  "gpt-4o-mini",   # cheaper, very capable
    "gpt35_turbo": "gpt-3.5-turbo", # even cheaper, faster — good baseline
}

# ── Step 5: evaluation judge ──────────────────────────────────────────────────
JUDGE_MODEL = "gpt-4o-mini"

CHUNK_SIZE    = 512
CHUNK_OVERLAP = 64
TOP_K         = 5

print("Config:")
print(f"  Embedding models : {list(EMBEDDING_MODELS.keys())}")
print(f"  LLM models       : {list(LLM_MODELS.keys())}")
print(f"  Judge LLM        : {JUDGE_MODEL}")
print(f"  Chunk size/overlap: {CHUNK_SIZE} / {CHUNK_OVERLAP}  |  Top-K: {TOP_K}")


Config:
  Embedding models : ['openai_small', 'bge_large']
  LLM models       : ['gpt4o_mini', 'gpt35_turbo']
  Judge LLM        : gpt-4o-mini
  Chunk size/overlap: 512 / 64  |  Top-K: 5


## 3 · Upload your PDFs

Run the cell below to upload PDF file(s) from your machine into `data/`.


In [4]:
try:
    from google.colab import files
    print("Click 'Choose Files' to upload one or more PDFs...")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dest = DATA_DIR / fname
        dest.write_bytes(data)
        print(f"  Saved: {dest}  ({len(data)/1024:.1f} KB)")
except ImportError:
    print("Not in Colab — place PDFs manually into the data/ folder.")

pdf_files = sorted(DATA_DIR.glob("*.pdf"))
print(f"\nPDFs in data/: {[p.name for p in pdf_files]}")
assert pdf_files, "❌  No PDFs found — upload at least one PDF before continuing."
print(f"✓ {len(pdf_files)} PDF(s) ready.")


Click 'Choose Files' to upload one or more PDFs...


Saving Final Policy document_LICs New Jeevan Shanti_V05_logo (1).pdf to Final Policy document_LICs New Jeevan Shanti_V05_logo (1).pdf
  Saved: data/Final Policy document_LICs New Jeevan Shanti_V05_logo (1).pdf  (575.1 KB)

PDFs in data/: ['Final Policy document_LICs New Jeevan Shanti_V05_logo (1).pdf']
✓ 1 PDF(s) ready.


## 4 · Step 1 — Parse PDFs with LlamaParse

LlamaParse sends each PDF to the LlamaCloud API and returns clean **Markdown**,
correctly handling multi-column layouts, tables, headers/footers, and footnotes —
much better than `pypdf` or `pdfminer` for structured documents.


In [5]:
import asyncio, json, nest_asyncio
from llama_parse import LlamaParse

nest_asyncio.apply()

parser = LlamaParse(
    api_key=os.environ["LLAMA_CLOUD_API_KEY"],
    result_type="markdown",
    verbose=True,
    num_workers=4,
)

async def parse_all():
    manifest = []
    for pdf_path in pdf_files:
        out_path = PARSED_DIR / f"{pdf_path.stem}.md"
        if out_path.exists():
            print(f"  [skip]    {pdf_path.name}  — already parsed → {out_path.name}")
            manifest.append({"source": str(pdf_path), "parsed": str(out_path)})
            continue

        print(f"  [parsing] {pdf_path.name} ...")
        docs = await parser.aload_data(str(pdf_path))
        text = "\n\n".join(d.text for d in docs)
        out_path.write_text(text, encoding="utf-8")
        print(f"  [done]    → {out_path.name}  ({len(text):,} chars)")
        manifest.append({"source": str(pdf_path), "parsed": str(out_path)})
    return manifest

manifest = await parse_all()
(PARSED_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(f"\n✓ Parsed {len(manifest)} document(s). Markdown files in parsed/")


/tmp/ipykernel_14099/2406996437.py:2: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


  [parsing] Final Policy document_LICs New Jeevan Shanti_V05_logo (1).pdf ...
Started parsing the file under job_id a02ff254-020b-4ec0-a53d-2ba9b31131c7
  [done]    → Final Policy document_LICs New Jeevan Shanti_V05_logo (1).md  (62,570 chars)

✓ Parsed 1 document(s). Markdown files in parsed/


## 5 · Steps 2 & 3 — Build a VectorStoreIndex per embedding model

We build **two indexes** from the same parsed text — one per embedding model —
so that retrieval quality can be compared fairly in Step 4.

- **`openai_small`** — calls the OpenAI Embeddings API (`text-embedding-3-small`)
- **`bge_large`** — runs entirely locally via `sentence-transformers` (no API cost)


In [6]:
from llama_index.core import (
    SimpleDirectoryReader, VectorStoreIndex,
    StorageContext, load_index_from_storage, Settings,
)
from llama_index.core.node_parser import SentenceSplitter

def get_embedding_model(key: str):
    cfg = EMBEDDING_MODELS[key]
    if cfg["type"] == "openai":
        from llama_index.embeddings.openai import OpenAIEmbedding
        return OpenAIEmbedding(model=cfg["model"], api_key=os.environ["OPENAI_API_KEY"])
    elif cfg["type"] == "local":
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding
        return HuggingFaceEmbedding(model_name=cfg["model"])
    raise ValueError(f"Unknown embedding type: {cfg['type']}")

def build_or_load_index(key: str, documents):
    print(f"\n── Embedding model: {key} ({EMBEDDING_MODELS[key]['model']}) ──")
    embed_model = get_embedding_model(key)
    Settings.embed_model = embed_model
    Settings.node_parser = SentenceSplitter(
        chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
    )

    persist_dir = INDEX_DIR / key
    if persist_dir.exists() and any(persist_dir.iterdir()):
        print(f"  Loading persisted index from {persist_dir}")
        ctx = StorageContext.from_defaults(persist_dir=str(persist_dir))
        return load_index_from_storage(ctx, embed_model=embed_model)

    print(f"  Building new index (chunking + embedding)...")
    idx = VectorStoreIndex.from_documents(documents, embed_model=embed_model, show_progress=True)
    idx.storage_context.persist(persist_dir=str(persist_dir))
    print(f"  ✓ Persisted to {persist_dir}")
    return idx

# Load parsed documents
documents = SimpleDirectoryReader(
    input_dir=str(PARSED_DIR), required_exts=[".md"]
).load_data()
print(f"Loaded {len(documents)} document chunk(s) from parsed/")

# Build one index per embedding model
indexes = {}
for key in EMBEDDING_MODELS:
    indexes[key] = build_or_load_index(key, documents)

print(f"\n✓ Indexes ready: {list(indexes.keys())}")


Loaded 1 document chunk(s) from parsed/

── Embedding model: openai_small (text-embedding-3-small) ──
  Building new index (chunking + embedding)...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/33 [00:00<?, ?it/s]

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## 6 · Define your evaluation questions

Edit this list to match your PDF content.


In [ ]:
QUESTIONS = [
    "What is the main topic of this document?",
    "Summarize the key findings in two sentences.",
    "What methodology was used?",
    "What are the main conclusions or recommendations?",
    # Add as many as you like
]
print(f"{len(QUESTIONS)} question(s) defined.")


## 7 · Step 4 — Compare LLMs during synthesis

For every **(embedding model × LLM)** combination we run all questions, record the
answer, the retrieved source chunks, and wall-clock latency.


In [ ]:
import time
from llama_index.llms.openai import OpenAI

def get_llm(name: str):
    return OpenAI(
        model=LLM_MODELS[name],
        api_key=os.environ["OPENAI_API_KEY"],
        temperature=0.1,
    )

results = []

for emb_key in EMBEDDING_MODELS:
    index = indexes[emb_key]
    for llm_key in LLM_MODELS:
        print(f"\n▶  embedding={emb_key}  |  llm={llm_key}")
        llm = get_llm(llm_key)
        qe  = index.as_query_engine(llm=llm, similarity_top_k=TOP_K)

        for q in QUESTIONS:
            t0 = time.time()
            try:
                resp   = qe.query(q)
                answer = str(resp).strip()
                sources = [
                    {"text": n.node.get_content()[:300], "score": round(n.score or 0, 4)}
                    for n in resp.source_nodes
                ]
                error = None
            except Exception as exc:
                answer, sources, error = None, [], str(exc)

            latency = round(time.time() - t0, 2)
            results.append({
                "question":        q,
                "embedding_model": emb_key,
                "llm_model":       llm_key,
                "answer":          answer,
                "sources":         sources,
                "latency_sec":     latency,
                "error":           error,
            })
            status = f"[ERROR: {error}]" if error else "✓"
            print(f"   {status}  {latency}s  Q: {q[:60]}...")

# Save raw responses
import json
raw_path = RESULTS_DIR / "raw_responses.json"
raw_path.write_text(json.dumps(results, indent=2))
print(f"\n✓ Saved {len(results)} raw responses → {raw_path}")


## 8 · Step 5 — Evaluate performance

Each answer is scored on two axes by a neutral judge LLM (`gpt-4o-mini`):

| Metric | What it measures |
|--------|-----------------|
| **Faithfulness** | Is the answer supported by the retrieved context? (catches hallucination) |
| **Relevancy** | Does the answer actually address the question? |
| **Latency** | Average wall-clock response time |
| **Error rate** | Fraction of calls that raised an exception |

The judge LLM is intentionally separate from the models being tested so scores
aren't biased toward whichever LLM generated the answer.


In [ ]:
import pandas as pd
from llama_index.core.evaluation import FaithfulnessEvaluator, RelevancyEvaluator
from llama_index.llms.openai import OpenAI

judge = OpenAI(model=JUDGE_MODEL, api_key=os.environ["OPENAI_API_KEY"], temperature=0)
faith_eval = FaithfulnessEvaluator(llm=judge)
relev_eval  = RelevancyEvaluator(llm=judge)

rows = []
total = len(results)
for i, r in enumerate(results, 1):
    print(f"  Evaluating {i}/{total} — emb={r['embedding_model']}  llm={r['llm_model']}  Q: {r['question'][:50]}...")
    row = {
        "question":        r["question"],
        "embedding_model": r["embedding_model"],
        "llm_model":       r["llm_model"],
        "answer":          r["answer"],
        "latency_sec":     r["latency_sec"],
        "error":           bool(r["error"]),
    }

    if r["error"] or not r["answer"]:
        row["faithfulness"] = None
        row["relevancy"]    = None
        rows.append(row)
        continue

    ctx = "\n\n".join(s["text"] for s in r["sources"])

    try:
        row["faithfulness"] = float(
            faith_eval.evaluate(query=r["question"], response=r["answer"], contexts=[ctx]).passing
        )
    except Exception as e:
        print(f"    ⚠ faithfulness eval error: {e}")
        row["faithfulness"] = None

    try:
        row["relevancy"] = float(
            relev_eval.evaluate(query=r["question"], response=r["answer"], contexts=[ctx]).passing
        )
    except Exception as e:
        print(f"    ⚠ relevancy eval error: {e}")
        row["relevancy"] = None

    rows.append(row)

detail_df = pd.DataFrame(rows)
detail_df.to_csv(RESULTS_DIR / "scorecard_detail.csv", index=False)
print(f"\n✓ Per-question detail saved → results/scorecard_detail.csv")


In [ ]:
# ── Summary scorecard ──────────────────────────────────────────────────────────
summary_df = (
    detail_df
    .groupby(["embedding_model", "llm_model"])
    .agg(
        avg_faithfulness = ("faithfulness", "mean"),
        avg_relevancy    = ("relevancy",    "mean"),
        avg_latency_sec  = ("latency_sec",  "mean"),
        error_rate       = ("error",        "mean"),
        n_questions      = ("question",     "count"),
    )
    .reset_index()
    .sort_values(["avg_faithfulness", "avg_relevancy"], ascending=False)
    .reset_index(drop=True)
)

summary_df.to_csv(RESULTS_DIR / "scorecard.csv", index=False)

# Pretty display
summary_df_display = summary_df.copy()
for col in ["avg_faithfulness", "avg_relevancy", "avg_latency_sec", "error_rate"]:
    summary_df_display[col] = summary_df_display[col].map(lambda x: f"{x:.3f}" if x == x else "—")

print("\n📊 Summary Scorecard (ranked best → worst):\n")
print(summary_df_display.to_string(index=False))
print("\n✓ Summary saved → results/scorecard.csv")
summary_df


## 9 · Download results

In [ ]:
try:
    from google.colab import files
    for fname in ["scorecard.csv", "scorecard_detail.csv", "raw_responses.json"]:
        fpath = RESULTS_DIR / fname
        if fpath.exists():
            files.download(str(fpath))
            print(f"  Downloading {fname}")
except ImportError:
    print("Not in Colab — results are in the results/ folder.")
